# 1. NN creation

### 1.1. Configuration & Setup

In [ ]:
import os
import time
import joblib
import itertools
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError, Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SINGLE_CONFIG = {
    # --- Paths ---
    "data_file"          : "AandB_output.txt",
    "model_file"         : "emulator.keras",
    "scalers_dir"        : "scalers/",
    "plots_dir"          : "plots/",

    # --- Data columns ---
    # File: [z, k1, k2, x12, om, log10fR0, A, A', B, B']
    "input_cols"         : [0, 1, 2, 3, 4, 5],
    "output_cols"        : [6, 7, 8, 9],
    "input_labels"       : ["z", "k1", "k2", "x12", "Ωm", "log10fR0"],
    "output_labels"      : ["A", "A'", "B", "B'"],

    # --- Split ---
    "val_size"           : 0.2,

    # --- Architecture ---
    "n_layers"           : 6,
    "n_neurons"          : 256,
    "activation"         : "swish",
    "output_activation"  : "linear",
    "delta"              : 1.5,

    # --- Training ---
    "batch_size"         : 1024,
    "epochs"             : 1000,
    "learning_rate"      : 1e-3,

    # --- Callbacks ---
    "early_patience"     : 50,
    "lr_patience"        : 20,
    "lr_factor"          : 0.5,
    "min_lr"             : 1e-7,

    # --- GPU ---
    "use_mixed_precision": True,

    # --- Reproducibility ---
    "seed"               : 42,
}

AUTOMATIC_CONFIG = {
    # --- Paths ---
    "data_file"          : "AandB_output.txt",
    "best_model_file"    : "best_emulator.keras",
    "scalers_dir"        : "best_scalers/",
    "plots_dir"          : "best_plots/",
    "log_file"           : "search_log.txt",

    # --- Data columns ---
    "input_cols"         : [0, 1, 2, 3, 4, 5],
    "output_cols"        : [6, 7, 8, 9],
    "input_labels"       : ["z", "k1", "k2", "x12", "Ωm", "log10fR0"],
    "output_labels"      : ["A", "A'", "B", "B'"],

    # --- Split ---
    "val_size"           : 0.2,

    # --- Architecture ---
    "output_activation"  : "linear",

    # --- Fixed training settings ---
    "batch_size"         : 1024,
    "epochs"             : 1000,

    # --- Callbacks ---
    "early_patience"     : 50,
    "lr_patience"        : 20,
    "lr_factor"          : 0.5,
    "min_lr"             : 1e-7,

    # --- GPU ---
    "use_mixed_precision": True,

    # --- Reproducibility ---
    "seed"               : None,

    # --- Search ---
    "max_runs"           : 100,
    "wall_time_seconds"  : 2 * 3600,
    "safety_margin"      : 5 * 60,
    "success_threshold"  : 1,

    # --- Search space ---
    "search_n_layers"    : [6, 7, 8],
    "search_n_neurons"   : [512, 1024],
    "search_activation"  : ["swish", "gelu"], # "relu", "tanh"
    "search_lr"          : [1e-3, 2e-3, 5e-3],
    "search_delta"       : [1.5, 2, 2.5]
}

### 1.2. Reproducibility

In [ ]:
def set_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)

### 1.3. GPU Configuration

In [1]:
def configure_gpu(use_mixed_prec):
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            # Prevent TF from grabbing all VRAM at startup
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)
        print(f"GPUs detected: {[g.name for g in gpus]}")

    if use_mixed_prec:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision: mixed_float16 enabled")

### 1.4. Data fetch

In [ ]:
def load_data(filepath, input_cols, output_cols):
    raw = np.loadtxt(filepath)
    X   = raw[:, input_cols]
    Y   = raw[:, output_cols]
    print(f"  Loaded {X.shape[0]} samples")
    print(f"  X shape: {X.shape}  |  Y shape: {Y.shape}")
    return X, Y

### 1.5. Preprocessing

In [ ]:
def preprocess(X, Y, val_size, seed):
    X_np, Y_np = np.array(X), np.array(Y)

    X_train, X_val, Y_train, Y_val = train_test_split(
        X_np, Y_np, 
        test_size=val_size, 
        random_state=seed
    )

    scaler_X = StandardScaler().fit(X_train)
    scaler_Y = StandardScaler().fit(Y_train)
    #scaler_Y = MinMaxScaler().fit(Y_train)

    X_train = scaler_X.transform(X_train)
    X_val   = scaler_X.transform(X_val)
    Y_train = scaler_Y.transform(Y_train)
    Y_val   = scaler_Y.transform(Y_val)

    print(f"  Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}")
    return X_train, X_val, Y_train, Y_val, scaler_X, scaler_Y

### 1.6. Model

In [ ]:
def build_and_compile(n_inputs, n_outputs, n_layers, n_neurons,
                      activation, output_activation, learning_rate, delta):
    model = Sequential()
    model.add(Input(shape=(n_inputs,)))

    for _ in range(n_layers):
        model.add(Dense(n_neurons, activation=activation))
        model.add(Dropout(0.01))

    model.add(Dense(n_outputs, activation=output_activation, dtype="float32"))

    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss=Huber(delta))
    return model

### 1.7. Callbacks

In [ ]:
def get_callbacks(cfg):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=cfg["early_patience"],
            restore_best_weights=True,
            verbose=0,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=cfg["lr_factor"],
            patience=cfg["lr_patience"],
            min_lr=cfg["min_lr"],
            verbose=0,
        ),
    ]

### 1.8. Training

In [ ]:
def train(model, X_train, Y_train, X_val, Y_val, cfg):
    cbs = get_callbacks(cfg)

    t0 = time.time()
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        callbacks=cbs,
        verbose=0,
    )
    print(f"\nTraining time: {time.time() - t0:.2f} s")
    return history

### 1.9. Evaluation

In [ ]:
def evaluate(model, X_val, Y_val, scaler_Y, output_labels):
    Y_pred_scaled = model.predict(X_val, verbose=0)

    Y_true = scaler_Y.inverse_transform(Y_val)
    Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)

    ref_scale = np.mean(np.abs(Y_true), axis=0, keepdims=True)
    ref_scale = np.where(ref_scale < 1e-10, 1.0, ref_scale)

    delta_pct = (Y_pred - Y_true) / ref_scale * 100.0

    print("\n  Percentage difference on validation set (ΔY%):")
    for i, label in enumerate(output_labels):
        col    = delta_pct[:, i]
        median = np.median(col)
        std    = np.std(col)
        p95    = np.percentile(np.abs(col), 95)
        p99    = np.percentile(np.abs(col), 99)
        status = "PASS" if p99 < 1.0 else "REVIEW"
        print(f"    {label:>4s}:  median={median:.4f}%  |  std={std:.4f}%  |  "
              f"|95th|={p95:.4f}%  |  |99th|={p99:.4f}%  [{status}]")

    return Y_true, Y_pred, delta_pct

### 1.10. Plotting

In [ ]:
def plot_loss(history, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(history.history["loss"],     "r--", lw=1.5, label="Train")
    ax.semilogy(history.history["val_loss"], "b-",  lw=1.5, label="Validation")
    ax.set_xlabel("Epochs", fontsize=13)
    ax.set_ylabel("Loss",   fontsize=13)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Loss plot saved: {save_path}")


def plot_percentage_difference(delta_pct, output_labels, save_path):
    """
    For each output, plot a two-panel figure:
      - Top:    histogram of ΔY%
      - Bottom: box-style summary (median, std, percentiles printed)
    Mimics the residual-panel style of the CLASS vs Effort reference plot.
    """
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for i, (ax, label) in enumerate(zip(axes.flat, output_labels)):
        col    = delta_pct[:, i]
        median = np.median(col)
        std    = np.std(col)

        # --- Histogram ---
        ax.hist(col, bins=1000, color="steelblue", edgecolor="black",
                alpha=0.75, density=True)

        # Reference lines
        ax.axvline(0.0,    color="black", ls="-",  lw=1.2, label="Zero")
        ax.axvline(median, color="green", ls="-",  lw=1.8,
                   label=f"Median: {median:.4f}%")
        ax.axvline(median + std, color="orange", ls="--", lw=1.2,
                   label=f"±1σ: {std:.4f}%")
        ax.axvline(median - std, color="orange", ls="--", lw=1.2)

        # Symmetric x-limits so the distribution is centred
        xlim = max(np.percentile(np.abs(col), 99.5), 1e-6) * 1.2
        ax.set_xlim(-xlim, xlim)

        ax.set_xlabel(r"$\Delta$" + f"{label} (%)", fontsize=12)
        ax.set_ylabel("Density", fontsize=12)
        ax.set_title(label, fontsize=14, fontweight="bold")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.25)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Percentage difference plot saved: {save_path}")

### 1.11. Save Artifacts

In [ ]:
def save_artifacts(model, scaler_X, scaler_Y, cfg):
    os.makedirs(cfg["scalers_dir"], exist_ok=True)

    model.save(cfg["model_file"])
    joblib.dump(scaler_X, os.path.join(cfg["scalers_dir"], "scaler_X.pkl"))
    joblib.dump(scaler_Y, os.path.join(cfg["scalers_dir"], "scaler_Y.pkl"))

    print(f"  Model  → {cfg['model_file']}")
    print(f"  Scalers → {cfg['scalers_dir']}")

### 1.12. Main function

In [ ]:
def main(cfg):

    set_seeds(cfg["seed"])
    configure_gpu(cfg["use_mixed_precision"])

    # ── Data ──
    print("Loading data...")
    X, Y = load_data(cfg["data_file"], cfg["input_cols"], cfg["output_cols"])

    # ── Preprocess ──
    print("\nPreprocessing...")
    X_train, X_val, Y_train, Y_val, scaler_X, scaler_Y = preprocess(
        X, Y, 
        cfg["val_size"], 
        cfg["seed"]
    )

    # ── Build & compile ──
    print("\nBuilding model...")
    n_in  = len(cfg["input_cols"])
    n_out = len(cfg["output_cols"])
    model = build_and_compile(n_in, 
                              n_out, 
                              cfg["n_layers"], 
                              cfg["n_neurons"],
                              cfg["activation"], 
                              cfg["output_activation"],
                              cfg["learning_rate"],
                              cfg["delta"]
                             )
    model.summary()

    # ── Train ──
    print("\nTraining...")
    history = train(model, X_train, Y_train, X_val, Y_val, cfg)

    # ── Evaluate ──
    print("\nEvaluating...")
    Y_true, Y_pred, delta_pct = evaluate(
        model, X_val, Y_val, scaler_Y, cfg["output_labels"]
    )

    # ── Plots ──
    os.makedirs(cfg["plots_dir"], exist_ok=True)
    plot_loss(history,
              os.path.join(cfg["plots_dir"], "loss_curves.pdf")
             )
    plot_percentage_difference(delta_pct,
                               cfg["output_labels"],
                               os.path.join(cfg["plots_dir"], "percentage_difference.pdf")
                              )

    # ── Save ──
    print("\nSaving...")
    save_artifacts(model, scaler_X, scaler_Y, cfg)

    print("\nDone.")

# =====================================

# 2. NN automatic training

### 2.1. Randomized architecture and hyperparameters

In [ ]:
def generate_search_space(cfg):
    all_combos = list(itertools.product(
        cfg["search_n_layers"],
        cfg["search_n_neurons"],
        cfg["search_activation"],
        cfg["search_lr"],
        cfg["search_delta"]
    ))

    rng = np.random.RandomState(cfg["seed"])
    rng.shuffle(all_combos)
    selected = all_combos[:cfg["max_runs"]]

    print(f"  Total possible combinations: {len(all_combos)}")
    print(f"  Selected for search: {len(selected)}")
    return selected

### 2.2. Save run info

In [ ]:
def log_run(log_path, run_id, params, worst_error, stats, elapsed, is_best):
    n_layers, n_neurons, activation, lr, delta = params

    with open(log_path, "a") as f:
        tag = " *** BEST ***" if is_best else ""
        f.write(f"{'='*70}\n")
        f.write(f"Run {run_id}{tag}\n")
        f.write(f"  Layers={n_layers}  Neurons={n_neurons}  "
                f"Activation={activation}  LR={lr}  Delta={delta}\n")
        f.write(f"  Training time: {elapsed:.2f}s\n")
        f.write(f"  Worst p99 |ΔY%|: {worst_error:.6f}%\n")
        for label, s in stats.items():
            f.write(f"    {label:>4s}: median={s['median']:.4f}%  "
                    f"std={s['std']:.4f}%  max={s['max_abs']:.4f}%  "
                    f"p95={s['p95']:.4f}%  p99={s['p99']:.4f}%\n")
        f.write(f"\n")

In [ ]:
def automatic_evaluate(model, X_val, Y_val, scaler_Y, output_labels):
    """Compute ΔY% and return max absolute error, per-output stats, and delta_pct."""
    Y_pred = model.predict(X_val, verbose=0)

    Y_true = scaler_Y.inverse_transform(Y_val)
    Y_pred = scaler_Y.inverse_transform(Y_pred)

    ref_scale = np.mean(np.abs(Y_true), axis=0, keepdims=True)
    ref_scale = np.where(ref_scale < 1e-10, 1.0, ref_scale)

    delta_pct = (Y_pred - Y_true) / ref_scale * 100.0

    stats = {}
    for i, label in enumerate(output_labels):
        col = delta_pct[:, i]
        stats[label] = {
            "median"  : np.median(col),
            "std"     : np.std(col),
            "max_abs" : np.max(np.abs(col)),
            "p95"     : np.percentile(np.abs(col), 95),
            "p99"     : np.percentile(np.abs(col), 99),
        }

    worst_error = max(s["p99"] for s in stats.values())
    return worst_error, stats, delta_pct

In [ ]:
def check_wall_time(start_time, wall_time, safety_margin):
    """Return True if there is enough time for another run."""
    elapsed = time.time() - start_time
    remaining = wall_time - elapsed
    return remaining > safety_margin

In [ ]:
def run_search(cfg):
    """Main search loop."""

    set_seeds(cfg["seed"])
    configure_gpu(cfg["use_mixed_precision"])

    # --- Load & preprocess (once) ---
    print("Loading data...")
    X, Y = load_data(cfg["data_file"], cfg["input_cols"], cfg["output_cols"])

    print("\nPreprocessing...")
    X_train, X_val, Y_train, Y_val, scaler_X, scaler_Y = preprocess(
        X, Y, 
        cfg["val_size"], 
        cfg["seed"]
    )

    # --- Search space ---
    print("\nGenerating search space...")
    combos = generate_search_space(cfg)

    # --- Prepare log ---
    log_path = cfg["log_file"]
    with open(log_path, "w") as f:
        f.write(f"Hyperparameter Search Log\n")
        f.write(f"Seed: {cfg['seed']}\n")
        f.write(f"Success threshold: {cfg['success_threshold']}%\n")
        f.write(f"Max runs: {cfg['max_runs']}\n")
        f.write(f"{'='*70}\n\n")

    # --- Search loop ---
    n_in  = len(cfg["input_cols"])
    n_out = len(cfg["output_cols"])

    best_worst_error   = np.inf
    best_params        = None
    best_delta_pct     = None
    best_history       = None

    job_start = time.time()
    success   = False

    for run_id, params in enumerate(combos, start=1):
        n_layers, n_neurons, activation, lr, delta = params

        # --- Wall-time check ---
        if not check_wall_time(job_start, cfg["wall_time_seconds"],
                               cfg["safety_margin"]):
            print(f"\nWall-time safety margin reached. Stopping search.")
            break

        print(f"\n--- Run {run_id}/{len(combos)}: layers={n_layers} "
              f"neurons={n_neurons} act={activation} lr={lr} Delta={delta}---")

        # Reset seed so weight init is reproducible per run
        set_seeds(cfg["seed"])

        model = build_and_compile(n_in, 
                                  n_out, 
                                  n_layers, 
                                  n_neurons,
                                  activation, 
                                  cfg["output_activation"],
                                  lr,
                                  delta
                                 )

        t0 = time.time()
        history = train(model, X_train, Y_train, X_val, Y_val, cfg)
        elapsed = time.time() - t0

        worst_error, stats, delta_pct = automatic_evaluate(
            model, X_val, Y_val, scaler_Y, cfg["output_labels"]
        )

        is_best = worst_error < best_worst_error
        if is_best:
            best_worst_error   = worst_error
            best_params        = params
            best_delta_pct     = delta_pct
            best_history       = history

            os.makedirs(cfg["scalers_dir"], exist_ok=True)
            os.makedirs(cfg["plots_dir"],   exist_ok=True)
            model.save(cfg["best_model_file"])
            joblib.dump(scaler_X, os.path.join(cfg["scalers_dir"], "scaler_X.pkl"))
            joblib.dump(scaler_Y, os.path.join(cfg["scalers_dir"], "scaler_Y.pkl"))
            print(f"  ✅ New best! Worst p99 |ΔY%| = {worst_error:.4f}%")
        else:
            print(f"  Worst p99 |ΔY%| = {worst_error:.4f}%")

        log_run(log_path, run_id, params, worst_error, stats, elapsed, is_best)

        # --- Success check ---
        if worst_error < cfg["success_threshold"]:
            print(f"\n🎯 SUCCESS: All points below {cfg['success_threshold']}%!")
            success = True
            break

    if best_params is None:
        print("No run completed. Exiting.")
        return

    # --- Save best ---
    print(f"\n{'='*70}")
    if success:
        print(f"Search finished: SUCCESS")
    else:
        print(f"Search finished: best worst p99 |ΔY%| = {best_worst_error:.4f}%")

    n_layers, n_neurons, activation, lr, delta = best_params
    print(f"Best config: layers={n_layers} neurons={n_neurons} "
          f"act={activation} lr={lr}  Delta={delta}")

    # --- Append best summary to log ---
    with open(log_path, "a") as f:
        f.write(f"\n{'='*70}\n")
        f.write(f"BEST RUN SUMMARY\n")
        f.write(f"  Layers={n_layers}  Neurons={n_neurons}  "
                f"Activation={activation}  LR={lr}  Delta={delta}\n")
        f.write(f"  Seed: {cfg['seed']}\n")
        f.write(f"  Worst p99 |ΔY%|: {best_worst_error:.6f}%\n")
        f.write(f"  Success: {success}\n")
        f.write(f"{'='*70}\n")

    # --- Plot best ---
    if best_history is not None:
        plot_loss(
            best_history,
            os.path.join(cfg["plots_dir"], "best_loss_curves.pdf")
        )
    
        plot_percentage_difference(
            best_delta_pct,
            cfg["output_labels"],
            os.path.join(cfg["plots_dir"], "best_percentage_difference.pdf")
        )

    print("\nDone.")

# 3. Routines

### 3.1. Single or Automated training

In [ ]:
# Just one at time: main or run_search
if __name__ == "__main__":
#    main(SINGLE_CONFIG)
    run_search(AUTOMATIC_CONFIG)